# Exercise 1
## Maximizing the benefit
with restricted capacity in each team

In [4]:
#Exercise 2 - INNOVATIVE INITIATIVES
#Question 5 
# -> maximizing benefit
using GLPK, JuMP

J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]               # Resource consumption
C = [28, 32, 35]    # Team capacities

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

model = Model(GLPK.Optimizer)

@variable(model,x[T,J],Bin)
@objective(model, Max, sum( x[i,j] * b[j] for i=T,j=J))

@constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
@constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

for dep in dependencies
    @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
end

@constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
@constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) >= 3)

optimize!(model)

println("solver status: ", termination_status(model))
println("solution status: ", primal_status(model))
println("object value: ", objective_value(model))

print("\n")
for i=T
    for j=J
        print(value(x[i,j]))
        print(" ")
    end
    print("\n")
end

solver status: OPTIMAL
solution status: FEASIBLE_POINT
object value: 244.0

0.0 1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 1.0 
1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 
0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 1.0 0.0 


## Lagrangian Relaxation

We added one constraint to the objective function, by introducing an additional constant $u$
$$
Z(u) = max \sum_{i=1}^{3}\sum_{j=1}^{15} x_{ij}b_j + \sum_{i=1}^{3}\lambda_i (C_i - \sum_{i=1}^3 x_{ij} a_j)
$$
and then minimized for $u$
$$
\min_{u} Z(u) 
$$


In [15]:
#Exercise 2
#Question 6/7   
# Lagrange Dual solution with gradient-descent -> min_u(max_x Z(u))
using GLPK, JuMP
J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]               # Resource consumption
C = [28, 32, 35]    # Team capacities
lambda = [2.9,3.8,1.03]

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

lambda = zeros(3)
bestUpperBound = Inf
bestLambda = copy(lambda)
maxIter = 500 

for k=1:maxIter
    # OPTIMIZATION MODEL
    model = Model(GLPK.Optimizer)

    @variable(model,x[T,J],Bin)
    @objective(model, Max, 
        sum( x[i,j] * b[j] for i=T,j=J)
        +
        sum( lambda[i] * (C[i] - sum( x[i,j] * a[j] for j=J) ) for i=T))

    @constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
    # @constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

    for dep in dependencies
        @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
    end

    @constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
    @constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) == 3)

    optimize!(model)

    # UPDATING UPPER BOUND and LAMBDA
    UB = objective_value(model)
    if UB < bestUpperBound
        bestUpperBound = UB
        bestLambda = copy(lambda)
    end

    # CALC NEW LAMBDA 
    for i=T
        mu = 0.05/sqrt(k)
        lambda[i] = max(lambda[i]  -  mu * ( C[i] - sum( value(x[i,j]) * a[j] for j=J) ), 0 )
    end
    # println("solver status: ", termination_status(model))
    # println("solution status: ", primal_status(model))
    if k % 25 == 0        
        println(
        "k = ", k,
        "   UB = ", round(UB,digits=2),
        "   best UB = ", round(bestUpperBound,digits=2),
        "   lambda = ", round.(lambda,digits=2))
    end
end

println("BestLagrangianUB = ", bestUpperBound)
println("BestLambda = ", bestLambda)

k = 25   UB = 275.77   best UB = 263.2   lambda = [2.72, 2.45, 3.19]
k = 50   UB = 261.19   best UB = 258.27   lambda = [2.43, 2.92, 2.59]
k = 75   UB = 267.74   best UB = 253.75   lambda = [2.4, 2.64, 2.81]
k = 100   UB = 255.51   best UB = 251.61   lambda = [2.84, 2.44, 2.57]
k = 125   UB = 253.91   best UB = 249.6   lambda = [2.75, 2.53, 2.37]
k = 150   UB = 252.77   best UB = 248.16   lambda = [2.6, 2.76, 2.45]
k = 175   UB = 254.89   best UB = 248.16   lambda = [2.55, 2.69, 2.47]
k = 200   UB = 247.23   best UB = 247.23   lambda = [2.42, 2.74, 2.49]
k = 225   UB = 257.97   best UB = 246.74   lambda = [2.51, 2.6, 2.51]
k = 250   UB = 251.38   best UB = 246.74   lambda = [2.7, 2.55, 2.44]
k = 275   UB = 250.08   best UB = 246.74   lambda = [2.59, 2.43, 2.67]
k = 300   UB = 255.58   best UB = 246.46   lambda = [2.49, 2.59, 2.61]
k = 325   UB = 256.72   best UB = 245.64   lambda = [2.55, 2.48, 2.55]
k = 350   UB = 250.71   best UB = 245.64   lambda = [2.46, 2.63, 2.5]
k = 375   UB = 2

## LP-Relaxation
letting the variables take real-values, not just integers and thereby receiving an upper bound, which is generally infeasible

In [16]:
#Exercise 2
#Question 8 - LP-Relaxation (positive real valued)
using GLPK, JuMP

J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]               # Resource consumption
C = [28, 32, 35]    # Team capacities

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

model = Model(GLPK.Optimizer)

@variable(model,x[T,J]>=0)
@objective(model, Max, sum( x[i,j] * b[j] for i=T,j=J))

@constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
@constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

for dep in dependencies
    @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
end

@constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
@constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) >= 3)

optimize!(model)

println("solver status: ", termination_status(model))
println("solution status: ", primal_status(model))
println("object value: ", objective_value(model))

print("\n")
for i=T
    for j=J
        print(value(x[i,j]))
        print(" ")
    end
    print("\n")
end

solver status: OPTIMAL
solution status: FEASIBLE_POINT
object value: 254.89189189189193

0.33333333333333315 0.33333333333333315 0.33333333333333315 0.0 0.7817047817047819 0.0 0.0 1.0000000000000004 0.0 0.0 6.504336911945362e-17 0.015561015561015568 0.0 0.0 0.0 
0.0 0.0 0.0 0.33333333333333315 0.0 0.5945945945945946 0.0 0.0 0.0 0.5945945945945946 0.0 0.9844389844389849 0.0 0.0 0.5945945945945946 
0.0 0.0 0.0 0.0 0.21829521829521836 0.0 1.0000000000000002 0.0 0.5945945945945946 0.0 0.0 0.0 1.0 1.0 0.0 
